# Fit and evaluate the regressors of a transfer method on a target robot

This is the regressor stage of the transfer flows (see `DESIGN-Transfer.md`):
1. It encodes the `training_data` and the `testing_data` of the target robot with the SP of the run (`sp_experiment`, `sp_run`). The latents are cached in the data directory of the run.
2. For every K of `k_shots` (`null` means all frames), it fits each regressor of `regressors` on K random training frames (seeded by `seed`):
   - `mlp`: the 64-64 MLP of the paper, on standardized latents, for `regressor_steps` steps;
   - `ridge`: a ridge regression on standardized latents, with `ridge_lambda`;
   - `ridge_meta`: a ridge regression on the raw latents, with the λ meta-learned by the SP (D5).
3. It also computes the trivial baseline `mean`, which predicts the mean of the K training targets.

`results.csv` has one row per (K, regressor, joint), with the RMSE on the normalized targets and in physical units.

In [ ]:
import sys
sys.path.append("../..")

from exp_run_config import Config
Config.PROJECTNAME = "BerryPicker"

import math
import pathlib
import pandas as pd
import torch
import torch.nn.functional as F

from exprunflow.reproducibility import configure_deterministic_run
from robot.position_factory import position_fields
from visual_proprioception.visproprio_models import VisProprio_SimpleMLPRegression
from visual_proprioception.transfer.transfer_data import (
    foreground_masks, kshot_rows, load_frames, physical_errors)
from visual_proprioception.transfer.transfer_models import RidgeRegressor, ridge_fit
from visual_proprioception.transfer.transfer_sp import create_transfer_sp, encode_frames

In [ ]:
# *** Initialize the variables with default values
# *** This cell should be tagged as parameters
# *** If papermill is used, some of the values will be overwritten

# If it is set to discard-old, the exprun will be recreated from scratch
creation_style = "exist-ok"

experiment = "visual_proprioception_transfer"
run = "vp_single-vgg19-from-panda_to-g1_s0"

# If not None, set an external experiment path
expruns_path = None
# If not None, set an output path
results_path = None

In [ ]:
if expruns_path:
    Config().set_exprun_path(pathlib.Path(expruns_path))
if results_path:
    Config().set_results_path(pathlib.Path(results_path))

exp = Config().get_experiment(experiment, run, creation_style=creation_style)
seed = configure_deterministic_run(exp["seed"])
device = Config().runtime["device"]
spexp = Config().get_experiment(exp["sp_experiment"], exp["sp_run"])
exp_robot = Config().get_experiment(exp["robot_exp"], exp["robot_run"])
sp = create_transfer_sp(spexp)

In [ ]:
def latents(split):
    """The latents and the targets of a split, cached in the data directory of the run"""
    cache = pathlib.Path(exp["data_dir"], f"latents_{split}.pt")
    if cache.exists():
        return torch.load(cache, weights_only=True)
    frames = load_frames(exp[split], spexp["image_size"], exp_robot, exp["frame_step"])
    masks = None
    if spexp.get("robot_mask"):
        masks = foreground_masks(frames["images"], frames["demo_index"],
                                 spexp["robot_mask"]["threshold"], spexp["robot_mask"]["dilate"])
    value = {"z": encode_frames(sp, frames["images"], masks, exp["encode_batch_size"]), "y": frames["targets"]}
    torch.save(value, cache)
    return value


training = latents("training_data")
testing = latents("testing_data")
print(f'training {tuple(training["z"].shape)}, testing {tuple(testing["z"].shape)}')

In [ ]:
def fit_predict(regressor, z_train, y_train, z_test):
    """Fits the regressor on the training latents and predicts the test targets"""
    if regressor == "ridge_meta":
        return RidgeRegressor(ridge_fit(z_train, y_train, sp.log_lambda()))(z_test)
    mean = z_train.mean(0)
    std = z_train.std(0, unbiased=False).clamp_min(1e-6)
    z_train, z_test = (z_train - mean) / std, (z_test - mean) / std
    if regressor == "ridge":
        return RidgeRegressor(ridge_fit(z_train, y_train, math.log(exp["ridge_lambda"])))(z_test)
    if regressor != "mlp":
        raise ValueError(f"Unknown regressor {regressor}")
    torch.manual_seed(seed)
    model = VisProprio_SimpleMLPRegression({
        "encoding_size": z_train.shape[1], "output_size": y_train.shape[1],
        "regressor_hidden_size_1": exp["regressor_hidden_size"],
        "regressor_hidden_size_2": exp["regressor_hidden_size"]}).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=exp["learning_rate"])
    generator = torch.Generator().manual_seed(seed)
    z_train, y_train = z_train.to(device), y_train.to(device)
    n = z_train.shape[0]
    model.train()
    for _ in range(exp["regressor_steps"]):
        rows = torch.randint(n, (min(exp["batch_size"], n),), generator=generator).to(device)
        optimizer.zero_grad()
        loss = F.mse_loss(model(z_train[rows]), y_train[rows])
        loss.backward()
        optimizer.step()
    model.eval()
    with torch.no_grad():
        return model(z_test.to(device)).cpu()

In [ ]:
fields = position_fields(exp_robot)
rows = []
for k in exp["k_shots"]:
    selection = kshot_rows(training["z"].shape[0], k, seed)
    y_train = training["y"][selection]
    predictions = {"mean": y_train.mean(0, keepdim=True).expand_as(testing["y"])}
    for regressor in exp["regressors"]:
        predictions[regressor] = fit_predict(regressor, training["z"][selection], y_train, testing["z"])
    for regressor, prediction in predictions.items():
        rmse = ((prediction - testing["y"]) ** 2).mean(0).sqrt().tolist()
        for field, value, (physical, unit) in zip(fields, rmse, physical_errors(exp_robot, rmse)):
            rows.append({"target": exp["target"], "method": exp["method"], "regressor": regressor,
                         "seed": seed, "k": len(selection), "k_all": k is None, "joint": field,
                         "rmse": value, "rmse_physical": physical, "unit": unit, "sp_run": exp["sp_run"]})
results = pd.DataFrame(rows)
results.to_csv(pathlib.Path(exp["data_dir"], "results.csv"), index=False)
print(results.groupby(["regressor", "k"])["rmse"].mean().unstack().round(4))

In [ ]:
exp.done()